## Cálculo de similitudes y asignación de claves de la Muestra 3

Una vez concluido el proceso de preprocesamiento y homologación de la Muestra 3, se desarrolló un procedimiento para identificar las correspondencias entre los vehículos de la muestra y los registros del catálogo vehicular.

Al igual que en las muestras anteriores, se utilizaron **el coeficiente de Gower y las medidas de similitud textual Token Set Ratio y Token Sort Ratio**, con el propósito de evaluar las características disponibles de los vehículos y las semejanzas entre sus descripciones.

Debido a la menor cantidad de variables disponibles en la Muestra 3, se adaptó el procedimiento de comparación, utilizando una similitud global ponderada compuesta por Gower y la similitud textual de descripción. Posteriormente, se incorporó una segunda fase destinada exclusivamente a los vehículos que permanecieron sin clave asignada, mediante criterios complementarios de correspondencia vehicular.

### 1. Preparación de las variables para la comparación

Inicialmente, se recuperaron la Muestra 3 homologada y el catálogo vehicular, trabajando sobre copias de ambas bases de datos para preservar la información original.

Se estandarizó el formato de las variables `Modelo`, `Transmisión` y `Ocupantes`, con el propósito de garantizar su compatibilidad con las categorías del catálogo. En particular, las cantidades de ocupantes y los años-modelo se transformaron a valores enteros cuando fue posible, mientras que las categorías de transmisión se normalizaron a mayúsculas.

Asimismo, se construyó una descripción adicional para cada registro del catálogo mediante la concatenación de las siguientes variables:

- `MARCA`: fabricante del vehículo.
- `TIPO`: denominación o línea del vehículo.
- `VERSION`: versión o conjunto de características particulares.
- `TRANSMISIÓN`: tipo de transmisión del vehículo.

Para incorporar la transmisión, los códigos `A` y `S` se transformaron a `AUT` y `STD`, respectivamente. Las categorías que no especificaban una transmisión concreta, como `A/S`, no recibieron una terminación adicional.

Durante la construcción de las descripciones se sustituyeron los valores faltantes por cadenas vacías únicamente en las columnas auxiliares, se normalizaron los espacios entre palabras y se convirtieron los textos generados a mayúsculas.

La variable `descripcion` de la Muestra 3 se conservó sin modificaciones, manteniendo la información textual originalmente registrada.

### 2. Similitud de Gower

El coeficiente de Gower permite comparar observaciones a partir de múltiples variables, incluso cuando estas presentan distintas escalas de medición. Para este procedimiento se utilizaron **tres variables tratadas como categóricas nominales**, correspondientes al año-modelo, la transmisión y la capacidad de ocupantes del vehículo.

Las variables consideradas fueron:

| Variable de la muestra | Variable del catálogo |
|---|---|
| Modelo | MODELO |
| Transmisión | TRANSMISIÓN |
| Ocupantes | OCUPANTES |

A diferencia de las muestras anteriores, no se incorporaron las variables `Marca`, `motor` y `AC` al cálculo de Gower de esta primera fase, debido a que no se encontraban disponibles de manera independiente en la Muestra 3.

Aunque `Modelo` y `Ocupantes` contienen valores numéricos, ambas variables se trataron como categorías. Por lo tanto, únicamente se evaluó su coincidencia exacta, sin establecer grados de semejanza entre años o capacidades diferentes.

Para cada variable se asignó una similitud de 1 cuando los valores coincidían y de 0 cuando eran diferentes. Posteriormente, se calculó el promedio de las coincidencias observadas:

$$
S_G(i,j)=
\frac{\displaystyle\sum_{k=1}^{p}\delta_{ijk}s_{ijk}}
{\displaystyle\sum_{k=1}^{p}\delta_{ijk}}
$$

Donde:

- $S_G(i,j)$ representa la similitud de Gower entre los vehículos $i$ y $j$.
- $p=3$ es el número de variables categóricas consideradas.
- $s_{ijk}=1$ si ambos vehículos coinciden en la variable $k$, y $0$ en caso contrario.
- $\delta_{ijk}=1$ cuando ambos registros tienen información disponible para la variable $k$, y $0$ cuando alguno presenta un valor faltante.

Las tres variables recibieron la misma importancia dentro del cálculo.

Para facilitar las comparaciones, las categorías de ambas bases de datos se codificaron conjuntamente mediante `pd.factorize()`, identificando los valores faltantes con el código `-1`.

Cuando alguno de los vehículos presentaba información faltante en determinada variable, esta se excluía del cálculo, ajustando el denominador al número de variables comparables. Si ninguna variable estaba disponible, la similitud se consideraba indefinida (`NaN`).

Se seleccionó esta medida porque permite evaluar conjuntamente características relevantes para la identificación vehicular, particularmente el año-modelo, la transmisión y la capacidad de ocupantes, utilizando la información disponible en ambas bases de datos.

El resultado se encuentra entre 0 y 1, donde **1 representa coincidencia en todas las variables comparables y 0 representa ausencia de coincidencias**.

### 3. Similitud textual

Para comparar las descripciones vehiculares se utilizaron las medidas Token Set Ratio y Token Sort Ratio, implementadas mediante la biblioteca `RapidFuzz` de Python.

**Token Set Ratio**

Esta medida compara las cadenas a partir de las palabras que las componen, identificando los términos compartidos y aquellos que aparecen exclusivamente en cada texto.

Su principal ventaja consiste en reconocer semejanzas elevadas entre cadenas que contienen palabras en diferente orden o cantidades distintas de información.

No obstante, presenta una limitación importante: cuando una cadena contiene todas las palabras de otra, pero incorpora términos adicionales, Token Set Ratio puede asignar una similitud de 100 %, aun cuando las descripciones no sean completamente equivalentes.

**Token Sort Ratio**

Esta medida separa las cadenas en palabras, las ordena alfabéticamente y posteriormente compara los textos resultantes.

A diferencia de Token Set Ratio, mantiene una mayor sensibilidad ante la presencia de palabras adicionales o ausentes. Por lo tanto, permite reducir las puntuaciones de similitud que podrían resultar excesivamente elevadas al utilizar únicamente Token Set Ratio.

**Combinación de las similitudes textuales**

Considerando las características de ambas medidas, se decidió construir una similitud textual ponderada que aprovechara la capacidad de Token Set Ratio para identificar cadenas semejantes e incorporara Token Sort Ratio para penalizar parcialmente las diferencias derivadas de términos adicionales.

Ambas medidas producen puntuaciones entre 0 y 100, que fueron divididas entre 100 para expresarlas en el intervalo de 0 a 1.

La similitud textual se definió como:

$$
S_T(i,j)=0.75S_{\text{set}}(i,j)+0.25S_{\text{sort}}(i,j)
$$

Donde:

- $S_T(i,j)$ es la similitud textual combinada.
- $S_{\text{set}}(i,j)$ es la similitud obtenida mediante Token Set Ratio.
- $S_{\text{sort}}(i,j)$ es la similitud obtenida mediante Token Sort Ratio.

Se asignó una ponderación de **75 % a Token Set Ratio y 25 % a Token Sort Ratio**, otorgando mayor importancia a la identificación de palabras compartidas, pero reduciendo la posibilidad de obtener similitudes perfectas entre descripciones con información adicional.

Cuando alguna de las cadenas comparadas presentaba un valor faltante (`NaN`) o estaba vacía, la similitud textual se consideraba indefinida, en lugar de asignarle una puntuación de cero.

### 4. Construcción de la similitud global ponderada

Una vez definidas las medidas anteriores, se estableció una similitud global integrada por dos componentes:

| Componente | Ponderación |
|---|---:|
| Similitud de Gower | 50 % |
| Similitud textual de descripción | 50 % |
| **Total** | **100 %** |

La similitud global se calculó mediante:

$$
S_{\text{global}}(i,j)=0.50S_G(i,j)+0.50S_D(i,j)
$$

Donde:

- $S_G(i,j)$ corresponde a la similitud de Gower.
- $S_D(i,j)$ corresponde a la similitud textual entre las descripciones completas.

Se asignó un **50 % a Gower**, debido a que incorpora las características estructuradas disponibles de los vehículos, como el año-modelo, la transmisión y la capacidad de ocupantes.

Por otra parte, se otorgó un **50 % a la similitud de descripción**, considerando que esta variable contiene información adicional sobre la marca, la denominación comercial y las características particulares del vehículo que no se encuentran disponibles en columnas independientes.

De esta manera, se buscó equilibrar la importancia de las características categóricas con la información contenida en las descripciones.

A diferencia de las Muestras 1 y 2, no se incorporó una similitud independiente de `Tipo` en esta primera fase, debido a que dicha variable no se encontraba disponible como columna original de la Muestra 3.

**Tratamiento de componentes faltantes**

Cuando alguna de las dos similitudes era indefinida (`NaN`), la operación de suma ponderada también producía una similitud global indefinida.

Por lo tanto, era necesario contar con ambas puntuaciones para obtener una similitud global válida. Los registros que no podían evaluarse o que no superaban el umbral establecido permanecían sin clave asignada.

### 5. Filtrado textual y preselección de candidatos

Para reducir el número de comparaciones necesarias y concentrar el análisis en los vehículos con mayor semejanza descriptiva, se implementó un procedimiento de **preselección basado exclusivamente en Token Set Ratio**.

A diferencia de las Muestras 1 y 2, donde la preselección utilizó la combinación de Token Set Ratio y Token Sort Ratio, en esta muestra se optó por utilizar únicamente Token Set Ratio durante el filtrado inicial.

Esta decisión permitió identificar descripciones que compartían términos relevantes, incluso cuando existían diferencias en el orden de las palabras o en la cantidad de información.

Se estableció como criterio de preselección conservar **las 15 descripciones distintas con mayor similitud de Token Set Ratio**.

Esta cantidad representa el número máximo de descripciones seleccionadas, no necesariamente el número de registros individuales, debido a que una misma descripción puede encontrarse asociada a diferentes registros del catálogo.

Es importante destacar que **el filtrado mediante Token Set Ratio no determinó directamente la clave asignada**. Su finalidad fue identificar un conjunto reducido de candidatos para calcular posteriormente la similitud textual combinada y el coeficiente de Gower.

### 6. Procedimiento inicial de asignación de claves

Una vez definidas las medidas de similitud y el criterio de preselección, se implementó un algoritmo para identificar la correspondencia más semejante entre cada vehículo de la Muestra 3 y los registros del catálogo vehicular.

El procedimiento se desarrolló en el siguiente orden:

1. **Filtrado por ocupantes:** inicialmente, se seleccionaron exclusivamente los registros del catálogo cuya capacidad de ocupantes coincidía exactamente con la del vehículo de la muestra. Si el valor era faltante o no existían registros correspondientes, el vehículo permanecía sin clave asignada en esta fase.

2. **Comparación inicial de descripciones:** dentro del conjunto de vehículos con la misma capacidad de ocupantes, se identificaron las descripciones distintas y se calculó su similitud mediante Token Set Ratio.

3. **Preselección de candidatos:** se seleccionaron hasta 15 descripciones distintas con mayor similitud textual y se recuperaron todos los registros del catálogo asociados a ellas.

4. **Cálculo de la similitud textual combinada:** sobre los candidatos seleccionados, se calculó nuevamente la similitud de descripción, utilizando la combinación ponderada de Token Set Ratio y Token Sort Ratio previamente definida.

5. **Cálculo de Gower:** para cada registro candidato, se calculó la similitud de Gower utilizando las tres variables categóricas: `Modelo`, `Transmisión` y `Ocupantes`, excluyendo los valores faltantes de cada comparación.

6. **Cálculo de la similitud global:** se integraron las puntuaciones de Gower y descripción mediante las ponderaciones del 50 % y 50 %, respectivamente.

7. **Eliminación de duplicidades:** cuando varios registros del catálogo compartían una misma descripción generada, se conservó únicamente aquel con mayor similitud global.

8. **Selección del mejor candidato:** se ordenaron los registros restantes de mayor a menor similitud global y se seleccionó aquel con la puntuación más alta.

9. **Asignación de clave:** finalmente, se estableció un umbral de aceptación de **0.80**. Si la similitud global del mejor candidato era estrictamente superior a dicho valor, se asignaba al vehículo de la muestra el identificador `ID` correspondiente del catálogo. En caso contrario, permanecía sin clave asignada en esta fase.

Este procedimiento se aplicó a todas las observaciones de la Muestra 3, almacenando los identificadores obtenidos en la columna `ID_asignado`.

### 7. Segunda fase: asignación complementaria de claves

Con el propósito de incrementar la cobertura del claveteo, se incorporó una segunda fase dirigida exclusivamente a los vehículos que permanecieron sin clave asignada después del procedimiento inicial.

Debido a que la Muestra 3 no contaba originalmente con columnas independientes de `Marca` y `Tipo`, fue necesario recuperar esta información a partir de las descripciones vehiculares.

Para ello, se implementó un procedimiento de extracción y comparación basado en las denominaciones comerciales del catálogo. Posteriormente, se evaluaron criterios de similitud textual y proximidad del año-modelo, utilizando el valor comercial como criterio principal para seleccionar el registro definitivo.

El procedimiento se desarrolló de la siguiente manera:

1. **Extracción y homologación de Marca:** se analizó el comienzo de cada descripción para identificar la marca del vehículo, considerando su aparición en la primera o segunda posición del texto. Se utilizaron las denominaciones del catálogo y un conjunto de equivalencias para reconocer marcas registradas bajo distintas nomenclaturas. Las marcas compuestas se priorizaron frente a coincidencias parciales.

2. **Identificación de Tipo:** una vez reconocida la marca, se examinó la porción restante de la descripción para identificar la denominación comercial del vehículo. Inicialmente, se buscaron coincidencias exactas con los tipos correspondientes a dicha marca, priorizando las denominaciones más largas. Cuando no existía coincidencia exacta, se admitió una identificación aproximada únicamente si la similitud textual era igual o superior a **0.88** y superaba en al menos **0.08** la puntuación de la segunda mejor alternativa. Cuando no fue posible identificar un tipo suficientemente claro, el registro permaneció sin asignación.

3. **Filtrado por Marca y Tipo:** se recuperaron los registros del catálogo correspondientes a la marca identificada y se calculó la similitud textual entre el tipo extraído y las denominaciones de `TIPO`. Se conservaron únicamente los candidatos con una similitud igual o superior a **0.70**, utilizando la combinación de Token Set Ratio y Token Sort Ratio previamente definida.

4. **Comparación de Modelo:** se calculó la diferencia absoluta entre el año-modelo del vehículo de la muestra y el correspondiente a cada candidato del catálogo. Se estableció una tolerancia máxima de **cinco años**, permitiendo considerar vehículos de modelos cercanos cuando no existía coincidencia exacta. Los registros sin un año-modelo numérico válido o sin candidatos dentro de esta tolerancia permanecieron sin asignación.

5. **Comparación de descripciones:** entre los candidatos que cumplieron las condiciones anteriores, se calculó la similitud textual de las descripciones completas. Para ello, se aplicó nuevamente la combinación ponderada de Token Set Ratio y Token Sort Ratio, conservando únicamente los registros con una similitud igual o superior a **0.65**.

6. **Cálculo complementario de Gower:** se evaluó la coincidencia nominal de las variables `Marca`, `Tipo` y `Modelo`, asignándoles la misma importancia. La similitud se calculó mediante:

   $$
   S_{G,2}(i,j)=
   \frac{
   s_{\text{Marca}}(i,j)+
   s_{\text{Tipo}}(i,j)+
   s_{\text{Modelo}}(i,j)
   }{3}
   $$

   Donde cada componente toma el valor de 1 cuando existe coincidencia exacta y de 0 en caso contrario.

   En este cálculo, la marca ya presentaba una coincidencia por el filtrado previo. Por su parte, las coincidencias de `Tipo` y `Modelo` se evaluaron de manera estricta, por lo que una similitud textual aceptable o una diferencia de años dentro de la tolerancia no implicaban necesariamente una coincidencia exacta en Gower.

   A diferencia de la primera fase, esta medida se utilizó como criterio complementario de desempate y no como parte de una nueva similitud global ponderada.

7. **Selección mediante el valor comercial:** una vez identificados los candidatos que cumplían los criterios anteriores, se conservaron únicamente aquellos con un identificador válido y un valor comercial numérico positivo.

   Entre estos registros se seleccionó aquel con el **mayor valor comercial** del catálogo. Cuando existían empates, se aplicaron sucesivamente los siguientes criterios: mayor similitud de Gower complementaria, mayor similitud de descripción, menor diferencia de año-modelo y mayor similitud de `Tipo`.

8. **Asignación de clave:** finalmente, se recuperó el identificador `ID` del candidato seleccionado y se incorporó a la muestra. Cuando no fue posible encontrar un registro que cumpliera las condiciones establecidas, el vehículo permaneció sin clave asignada.

Esta segunda fase permitió ampliar las posibilidades de correspondencia para vehículos que no habían superado el procedimiento inicial, aprovechando la información comercial contenida en las descripciones.

Es importante destacar que **el valor comercial constituyó el criterio principal de selección únicamente después de aplicar los filtros de marca, tipo, modelo y descripción**. Por lo tanto, el procedimiento no consistió en seleccionar directamente el vehículo más costoso del catálogo, sino el de mayor valor comercial entre los candidatos considerados compatibles.

Las claves obtenidas durante la primera fase se conservaron sin modificaciones, aplicando este procedimiento exclusivamente a los registros pendientes de asignación.

### 8. Resultados y alcance del procedimiento

Una vez concluidas ambas fases, se generó el archivo `Muestra_3_claveteada.xlsx`, que contiene las variables de la muestra homologada junto con los identificadores asignados.

La metodología final combinó dos estrategias complementarias: **una primera fase basada en la similitud ponderada de Gower y descripción, y una segunda fase de asignación mediante criterios adicionales de correspondencia vehicular y selección por valor comercial**.

Las asignaciones obtenidas inicialmente se conservaron, mientras que el procedimiento complementario permitió evaluar nuevamente los registros que permanecieron sin clave.

Para facilitar la identificación y revisión de los resultados, se incorporaron las columnas `fase_asignacion` y `motivo_no_asignacion`, que permiten distinguir la etapa en la que se obtuvo cada clave y registrar las causas por las cuales determinados vehículos permanecieron sin asignación.

Adicionalmente, se conservaron variables auxiliares de auditoría relacionadas con la marca y el tipo recuperados, las similitudes calculadas durante la segunda fase, la diferencia de años y el valor comercial seleccionado.

El procedimiento generó tres archivos:

- `Muestra_3_solo_fase1.xlsx`: contiene las asignaciones originales de la primera fase.
- `Muestra_3_claveteada.xlsx`: contiene los resultados finales de ambas fases.
- `Auditoria_Muestra_3_no_asignados.xlsx`: contiene los registros que permanecieron sin clave después de aplicar ambos procedimientos.

Asimismo, se contabilizaron los vehículos asignados en cada fase, el total de registros identificados, aquellos que permanecieron sin asignación y el porcentaje final de cobertura.

Debe señalarse que las claves asignadas representan correspondencias determinadas mediante los criterios establecidos en cada fase y no garantizan una identificación exacta de todas las características del vehículo.

De esta manera, el procedimiento permitió ampliar la cobertura del claveteo, aprovechando tanto las características estructuradas como la información contenida en las descripciones, y manteniendo diferenciados los criterios utilizados en cada fase.